In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from datetime import datetime
import numpy as np

log_file_list = [
    '../work/experiments/stellar-core/scp_4_v2/tsm-sc-000/node1/stellar-core.log'
]

throughput_list = []
latency_list = []

# ---- Regex patterns ----
submit_pattern = re.compile(
    r"(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}).*"
    r"\[SCP SUBMIT\] Submitting (\d+) operations for ledger (\d+)"
)

commit_pattern = re.compile(
    r"(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}).*"
    r"\[SCP COMMIT\] Ledger (\d+):"
)

for log_file in log_file_list:
    print(f"Processing: {log_file}")

    submit_times = {}
    commit_times = {}
    ops_per_ledger = {}

    # ---- Parse logs ----
    with open(log_file) as f:
        for line in f:
            smatch = submit_pattern.search(line)
            if smatch:
                ts = datetime.strptime(smatch.group(1), "%Y-%m-%dT%H:%M:%S.%f")
                ops = int(smatch.group(2))
                ledger = int(smatch.group(3))

                # Keep first submit only
                if ledger not in submit_times:
                    submit_times[ledger] = ts
                    ops_per_ledger[ledger] = ops

            cmatch = commit_pattern.search(line)
            if cmatch:
                ts = datetime.strptime(cmatch.group(1), "%Y-%m-%dT%H:%M:%S.%f")
                ledger = int(cmatch.group(2))
                commit_times[ledger] = ts

    # ---- Compute latency records ----
    records = []
    for ledger, ctime in commit_times.items():
        if ledger in submit_times:
            latency = (ctime - submit_times[ledger]).total_seconds()
            records.append({
                "ledger": ledger,
                "commit_time": ctime,
                "latency": latency,
                "ops": ops_per_ledger.get(ledger, 0)
            })

    if not records:
        continue

    df = pd.DataFrame(records)

    # ---- Normalize time ----
    start_time = df["commit_time"].min()
    df["time_sec"] = df["commit_time"].apply(
        lambda t: (t - start_time).total_seconds()
    )
    df["time_sec_round"] = df["time_sec"].round().astype(int)

    # ---- Throughput (ops/sec) ----
    throughput = df.groupby("time_sec_round")["ops"].sum()

    full_index = np.arange(
        int(throughput.index.min()),
        int(throughput.index.max()) + 1
    )
    throughput = throughput.reindex(full_index, fill_value=0)

    # ---- Latency (avg per sec) ----
    latency_series = (
        df.groupby("time_sec_round")["latency"]
        .mean()
        .reindex(full_index, fill_value=0)
    )

    throughput_list.append(throughput)
    latency_list.append(latency_series)

    # # ---- Plot Throughput ----
    # plt.figure(figsize=(10, 4))
    # plt.plot(throughput.index, throughput.values, "-")
    # plt.xlabel("Time (s)")
    # plt.ylabel("Ops / sec")
    # plt.title("Stellar SCP Throughput vs Time")
    # plt.tight_layout()
    # plt.show()

    # # ---- Plot Latency ----
    # plt.figure(figsize=(10, 4))
    # plt.plot(latency_series.index, latency_series.values, "-", color="r")
    # plt.xlabel("Time (s)")
    # plt.ylabel("Latency (s)")
    # plt.title("SCP SUBMIT → SCP COMMIT Latency")
    # plt.tight_layout()
    # plt.show()

    # ---- Averages (steady state) ----
    if len(throughput) > 90:
        avg_tp = np.mean(throughput.values[-90:-30])
        avg_lat = np.mean(latency_series.values[-90:-30])
    else:
        avg_tp = np.mean(throughput.values)
        avg_lat = np.mean(latency_series.values)

    print("Average Throughput (ops/sec):", avg_tp)
    print("Average Latency (sec):", avg_lat)
    print("-" * 60)


Processing: ../work/experiments/stellar-core/scp_4_v2/tsm-sc-000/node1/stellar-core.log
Average Throughput (ops/sec): 1583.3333333333333
Average Latency (sec): 0.9769833333333334
------------------------------------------------------------
